# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
from pathlib import Path

load_dotenv("C:\\Users\\ioana\\Desktop\\curs AI Engineering\\echochamber-project-team-1\.env")

<>:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\ioana\AppData\Local\Temp\ipykernel_34784\2386137252.py:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  load_dotenv("C:\\Users\\ioana\\Desktop\\curs AI Engineering\\echochamber-project-team-1\.env")


True

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM (Large Language Model) este un model de inteligență artificială antrenat pe cantități masive de text, care îi permit să înțeleagă și să genereze limbaj uman coerent și relevant. Această capacitate îl face util pentru sarcini precum traducerea, rezumarea textelor, scrierea creativă și răspunsul la întrebări.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [6]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 5 ani.
Maximum 80 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
În ultimii 5 ani, politica românească a fost marcată de o instabilitate guvernamentală frecventă, cu multiple schimbări de premier și coaliții politice. De asemenea, s-a observat o accentuare a luptei anticorupție și o consolidare a rolului instituțiilor judiciare în viața publică.

--- Gemini 2.5 Flash ---
Ultimii cinci ani au adus o tranziție de la guverne minoritare sau de coaliție restrânsă la o coaliție guvernamentală largă PNL-PSD, menită să asigure stabilitate. Această perioadă a fost marcată și de apariția unor noi partide politice și de continuarea discuțiilor privind reforma sistemului de justiție.

--- OpenRouter Free ---
Guvernele au evoluat de la PSD la o coaliție PNL-USR-PMP, formată după alegerile parlamentare din 2021. Reformele judiciare și politica bazată pe fondurile UE au fost centralele schimbări politice.


## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [7]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii sunt implicați în corupție, și suntem conduși de un stat paralel care bagă bețe în roate oamenilor simpli."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Acuzator
Emoție dominantă: Furie/Dezamăgire
Țintă principală: Clasa politică și "statul paralel"
Populism: da

--- Gemini 2.5 Flash ---
Ton: Acuzator, conspirativ.
Emoție dominantă: Furie, neîncredere.
Țintă principală: Clasa politică și structurile de putere neoficiale.
Populism: da

--- OpenRouter Free ---
Ton: Conspirativ, alarmist  
Emoție dominantă: Frustrare / teamă  
Țintă principală: Politicienii și „statul paralel” (elita de putere)  
Populism: da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [8]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [9]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicieni', 'populism': True, 'explicatie_scurta': 'Comentariul exprimă o frustrare generală față de politicieni, acuzându-i de corupție și ignorarea voinței poporului, un discurs tipic populist.'}

--- Gemini 2.5 Flash ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicienii', 'populism': True, 'explicatie_scurta': "Comentariul exprimă furie și dezamăgire față de clasa politică, acuzând-o de corupție și de ignorarea poporului, folosind un limbaj populist prin contrastul 'oameni simpli' vs. 'politicieni'."}

--- OpenRouter Free ---
{'ton': 'negativ', 'emotie_dominanta': 'frica', 'tinta_principala': 'distrust in political class', 'populism': True, 'explicatie_scurta': 'Comentariul expresează o frustrare profundă față de corupție politică și o sentimente de impunitate, dar și de desconectare dintre politicieni și poporul. Suggerează o desconfianță totală în s

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [10]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Anularea alegerilor de către Curtea Constituțională poate duce la organizarea de noi alegeri, ceea ce implică o perioadă de incertitudine politică și o posibilă reconfigurare a forțelor politice. Acest proces poate influența stabilitatea guvernamentală și direcția politicilor publice pe termen scurt și mediu.

temperature=0.7:
Anularea alegerilor de către Curtea Constituțională poate duce la organizarea de noi alegeri, ceea ce implică o perioadă de incertitudine politică și posibile schimbări în configurația puterii. Acest proces poate, de asemenea, să genereze dezbateri despre legitimitatea procesului electoral și necesitatea unor reforme.

temperature=1.2:
Anularea alegerilor de către Curtea Constituțională poate duce la organizarea de noi alegeri, ceea ce înseamnă o perioadă de incertitudine politică și o potențială schimbare în componența legislativului sau executivului. Acea

In [12]:
response = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Rezumă în 3 propoziții neutre următorul text politic: Chiar exista un stat paralel in Romania?",
    system="Ești un analist politic neutru."
)
print(response)

Statul paralel este un concept disputat în spațiul public românesc, adesea invocat în discuțiile despre justiție și instituțiile statului. Susținătorii existenței sale invocă presupuse legături neoficiale între anumite grupuri de interese și structuri de putere, care ar influența deciziile statului. Criticii, pe de altă parte, consideră că acest concept este o construcție retorică lipsită de dovezi concrete, utilizată politic.


In [15]:
response = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Clasifică comentariul următor ca: pro-guvern / anti-guvern / neutru. Text: Bolojan crede ca ne poate pacali cu strategiile lui, dar stim ca e si el sorosist, HUOO!"
)
print(response)

Clasificare: **Anti-guvern**

**Justificare:**

Comentariul exprimă o atitudine negativă și suspicioasă față de "Bolojan" (probabil un politician sau o figură publică asociată cu guvernarea). Folosirea termenului "pacali" sugerează lipsă de încredere în intențiile sale. Asocierea cu "sorosist" este folosită într-un context peiorativ, sugerând o acuzație de manipulare sau de a fi sub influența unor forțe considerate negative. Exclamația "HUOO!" întărește tonul de respingere și animozitate.


In [20]:
for temp in [0.0, 0.7, 1.0]:
    response = ask(
        provider="gemini",
        model="gemini-2.5-flash-lite",
        prompt="Rezumă în 3 propoziții neutre următorul text politic: Chiar exista un stat paralel in Romania?",
        system="Ești un analist politic neutru.",
        temperature=temp
    )
    print(f"Temperature: {temp}\nResponse:\n{response}\n{'-'*40}")

Temperature: 0.0
Response:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]
----------------------------------------
Temperature: 0.7
Response:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]
----------------------------------------
Temperature: 1.0
Response:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]
----------------------------------------


## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | [da] / nu / parțial | [da] / nu / parțial | [da] / nu / parțial | [da] / nu | La 0.7 este fix cantitatea de rationalitate dar si sentiment uman necesara pentru proiect.|
| OpenRouter Free | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
| Llama / alt model testat | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
### Decizie
**Model principal ales:**  gemini-2.5-flash 
**Model de rezervă:**  gemini-2.5-flash-lite
**Temperature recomandată:**  0.7
**De ce am ales acest model?**  
Scrieți 2-3 propoziții. Menționați calitatea răspunsului, stabilitatea și dacă modelul poate fi folosit pentru adnotarea comentariilor.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [ ]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales